# BU7141: reproduce the team results from raw data

Open this notebook in the extracted project folder. This is a new reconstruction of the supplied analyses, not Yinran's original notebook. All processing is local. See README.md and METHODS.md for scope and definitions.

In [ ]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys
import pandas as pd

root = Path.cwd()
assert (root / 'run.py').is_file(), 'Open the notebook from the extracted project folder.'
print('Project:', root.name)

In [ ]:
# Dependencies must already be installed: python -m pip install -r requirements.txt
completed = subprocess.run([sys.executable, str(root / 'run.py')], cwd=root, check=True)
assert completed.returncode == 0

In [ ]:
validation = json.loads((root / 'outputs/validation_report.json').read_text(encoding='utf-8'))
display({k: validation[k] for k in ['status', 'source_rows', 'reference_sections', 'cells_compared', 'mismatches']})
display(pd.read_csv(root / 'outputs/validation/ReferenceComparison.csv')[['Table', 'Status', 'CellsCompared', 'Mismatches']])

In [ ]:
annual = pd.read_csv(root / 'outputs/monthly/AnnualSummary.csv')
display(annual[['Year', 'OpeningMemberships', 'NewEnrolments', 'Dropouts', 'ClosingMemberships']])
seasonal = pd.read_csv(root / 'outputs/age_groups/SeasonalComparison.csv')
display(seasonal[['MonthName', 'MeanAnnualEnrolmentShareA', 'MeanAnnualEnrolmentShareB', 'MeanDropoutRateA', 'MeanDropoutRateB']])

In [ ]:
retention = pd.read_csv(root / 'outputs/retention/Summary.csv')
all_years = retention[retention['Scope'].eq('All 2015-2018')].copy()
all_years['AdjustedRetentionA_pct'] = 100 * all_years['MatchedRateA']
all_years['AdjustedRetentionB_pct'] = 100 * all_years['MatchedRateB']
display(all_years[['HorizonDays', 'EligibleNA', 'EligibleNB', 'AdjustedRetentionA_pct', 'AdjustedRetentionB_pct', 'GapPP_AminusB']])
print('A = estimated entry age 18–24; B = estimated entry age 35+.')

In [ ]:
with sqlite3.connect(root / 'outputs/database/sports_centre.sqlite') as con:
    display(pd.read_sql_query('SELECT * FROM MemberSummary LIMIT 5', con))
    display(pd.read_sql_query('SELECT Activity_Name, COUNT(*) AS Members FROM Activity GROUP BY Activity_Name ORDER BY Members DESC', con))
    assert con.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
    assert con.execute('PRAGMA foreign_key_check').fetchall() == []

## Interpretation

The comparison verifies numerical reproduction of the provided Excel results. It does not recover individual invoices, renewal dates, intermediate membership gaps or causal intervention effects. Keep the historical recorded-age activity views separate from estimated-entry-age retention. The new SQLite implementation is an additional reproducibility step; it was not the database used by the team originally.